# Model-4 (CNN - Beat-by-beat data)

Use the Python 3.11 kernel and private preprocessed data; see [setup](../README.md). Run cells top to bottom.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from ai_cpet_ots.dataloaders import Model_4_Dataset, create_dataset_model_4
from ai_cpet_ots.helpers import collate, np
from ai_cpet_ots.models import Model_4
from ai_cpet_ots.paths import ECG_CONTROL_DIR, ECG_OTS_DIR, MODEL_WEIGHTS, output_path
from ai_cpet_ots.testers import test_model_4

In [ ]:
def optimize_model(X_train, Y_train):
    """Search the notebook's study hyperparameter bounds using training inputs only.

    Returns the optimizer result used by train_model. Requires the optimization
    extra and can be expensive; it is not run in the default evaluation workflow."""
    from ai_cpet_ots.optimizers import Optimizer_Model_4

    opt = Optimizer_Model_4(X_train, Y_train)

    bounds = [
        {"name": "n_epochs", "type": "discrete", "domain": np.arange(5, 25, 5)},
        {"name": "dropout", "type": "discrete", "domain": np.arange(0, 1, 0.1)},
        {"name": "lr", "type": "discrete", "domain": (1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1)},
        {
            "name": "weight_decay",
            "type": "discrete",
            "domain": (0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1),
        },
        {"name": "batch_size", "type": "discrete", "domain": [8, 16, 32]},
    ]

    _, opt_params_gpbo = opt.run_bo_gp(bounds, 10)

    return opt_params_gpbo

In [ ]:
def train_model(opt_params, X_train, Y_train):
    """Train this notebook's model from opt_params and the supplied training arrays.

    Keeps the original objective, optimizer, and epoch settings. Returns the
    trained model; checkpoint saving is controlled separately by save_model."""
    if torch.cuda.is_available():
        device = torch.device("cuda:0")
    else:
        device = torch.device("cpu")

    n_epochs, dropout, lr, weight_decay, batch_size, threshold = opt_params

    print(
        f"PARAMS n_epochs: {n_epochs}, dropout: {dropout:.1f}, lr: {lr}, weight_decay: {weight_decay}, batch_size: {batch_size}, threshold {threshold:.3f}"
    )

    trainset = Model_4_Dataset(X_train, Y_train)
    trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, collate_fn=collate)

    model = Model_4(dropout, threshold).to(device)

    pos_samples = np.sum(Y_train)
    neg_samples = len(Y_train) - pos_samples
    pos_weight = torch.tensor([neg_samples / pos_samples]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    for e in range(n_epochs):
        train_loss = 0

        for batch in trainloader:
            x_batch, y_batch, _ = batch
            x_batch, y_batch = x_batch.to(device), torch.tensor(y_batch).to(device)

            y_pred, _ = model(x_batch)

            loss = criterion(y_pred.flatten(), y_batch.float())
            loss.backward()

            optimizer.step()
            optimizer.zero_grad()

            train_loss += loss.item()

        print(f"Epoch [{e + 1}/{n_epochs}] - train_loss {train_loss / len(trainloader):.4f}")

    return model

In [ ]:
def test_model(X_test, Y_test, model=None):
    """Load the published checkpoint when model is None and evaluate test inputs.

    Constructs the study DataLoader and forwards plot options to the reusable
    evaluator. Private inputs must use the original feature order and scaling."""
    if torch.cuda.is_available():
        device = torch.device("cuda:0")
    else:
        device = torch.device("cpu")

    if model is None:
        model = Model_4().to(device)
        model.load_state_dict(
            torch.load(MODEL_WEIGHTS / "Model_4.pth", map_location=device, weights_only=True)
        )

    testset = Model_4_Dataset(X_test, Y_test)
    testloader = DataLoader(testset, batch_size=32, collate_fn=collate)
    auroc, auprc, f1, acc = test_model_4(model, "Model 4", testloader, True, True, True, True)
    print(
        f"AUROC: {auroc:.3f}, AUCPR: {auprc:.3f}, F1: {f1:.3f}, ACC: {acc:.3f}, Threshold {model.threshold:.3f}"
    )

In [ ]:
training_model = False
save_model = False
model_name = "Model_4"

In [ ]:
"""Load beat-by-beat data"""
X_train, Y_train, X_test, Y_test = create_dataset_model_4(
    ECG_CONTROL_DIR, ECG_OTS_DIR, filter_mode="pan-tompkins"
)

In [ ]:
if training_model:
    print("Optimizing model...")
    opt_params = optimize_model(X_train, Y_train)

    print("Training model...")
    model = train_model(opt_params, X_train, Y_train)

    print("Testing model...")
    test_model(X_test, Y_test, model)

    if save_model:
        torch.save(model.state_dict(), output_path(f"{model_name}.pth"))
else:
    print("Testing model...")
    test_model(X_test, Y_test)